In [71]:
import os
import pandas as pd
import numpy as np

In [50]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [51]:
df = pd.read_excel('../data/Online Retail.xlsx')


In [52]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# Business Understanding

# Data Cleaning

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [54]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [55]:
df.shape

(541909, 8)

In [56]:
print("Duplicate rows: ", df.duplicated().sum())

Duplicate rows:  5268


In [57]:
df = df.drop_duplicates()

In [58]:
df.shape

(536641, 8)

In [59]:
print(df.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64


In [60]:
# removing all the rows w/ missing customerID
df = df.dropna(subset=["CustomerID"])

In [61]:
print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [62]:
# removing cancellation/returned items
df = df[~df["InvoiceNo"].astype(str).str.upper().str.startswith("C")]

In [63]:
# removing invalid transactions (invalid prices or quantities)
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]


In [64]:
# final statistics + check (doing it before dropping non relevant columns)
print(df.dtypes)
print(df.isnull().sum())
print("Duplicate rows: ", df.duplicated().sum())
print("Shape: ", df.shape)
df["CustomerID"] = df["CustomerID"].astype(int)


InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64
Duplicate rows:  0
Shape:  (392692, 8)


In [65]:
#dropping non relevant columns
df = df.drop(columns=["Description", "Country"])


In [66]:
print("Final shape: ", df.shape)
print("\nMissing values: ")
print(df.isnull().sum())
print("\nData types: ")
print(df.dtypes)

Final shape:  (392692, 6)

Missing values: 
InvoiceNo      0
StockCode      0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
dtype: int64

Data types: 
InvoiceNo              object
StockCode              object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
dtype: object


# Data Exploration

# Data Visualization

#Data Visualization


# Customer Feature Engineering

In [77]:
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

In [78]:
# recency, frequency, monetary, total quantity of items customers bought
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (df["InvoiceDate"].max() - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
    total_quantity=('Quantity', 'sum')
).reset_index()
rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,total_quantity
0,12346,325,1,77183.60,74215
1,12347,1,7,4310.00,2458
2,12348,74,4,1797.24,2341
3,12349,18,1,1757.55,631
4,12350,309,1,334.40,197
5,12352,35,8,2506.04,536
6,12353,203,1,89.00,20
7,12354,231,1,1079.40,530
8,12355,213,1,459.40,240
9,12356,22,3,2811.43,1591


In [79]:
rfm["average_order_value"] = rfm['Monetary'] / rfm['Frequency'] 
rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,total_quantity,average_order_value
0,12346,325,1,77183.60,74215,77183.600000
1,12347,1,7,4310.00,2458,615.714286
2,12348,74,4,1797.24,2341,449.310000
3,12349,18,1,1757.55,631,1757.550000
4,12350,309,1,334.40,197,334.400000
5,12352,35,8,2506.04,536,313.255000
6,12353,203,1,89.00,20,89.000000
7,12354,231,1,1079.40,530,1079.400000
8,12355,213,1,459.40,240,459.400000
9,12356,22,3,2811.43,1591,937.143333


In [80]:
# product diversity / breadth features

# unique products purchased per customer
unique_products = df.groupby('CustomerID')['StockCode'].nunique().rename('unique_products').reset_index()
rfm = rfm.merge(unique_products, on='CustomerID', how='left')

# product diversity ratio = unique products / total quantity purchased
rfm['product_diversity_ratio'] = rfm['unique_products'] / rfm['total_quantity']
rfm['product_diversity_ratio'] = rfm['product_diversity_ratio'].replace([np.inf, -np.inf], np.nan)

# repeat purchase rate = % of a customer's distinct products bought more than once
product_counts = df.groupby(['CustomerID', 'StockCode']).size().reset_index(name='purchase_count')
repeat_rate = (
    product_counts.groupby('CustomerID')['purchase_count']
    .apply(lambda x: (x > 1).mean())
    .rename('repeat_purchase_rate')
    .reset_index()
)
rfm = rfm.merge(repeat_rate, on='CustomerID', how='left')

rfm.head(20)

,CustomerID,Recency,Frequency,Monetary,total_quantity,average_order_value,unique_products,product_diversity_ratio,repeat_purchase_rate
0,12346,325,1,77183.60,74215,77183.600000,1,0.000013,0.000000
1,12347,1,7,4310.00,2458,615.714286,103,0.041904,0.417476
2,12348,74,4,1797.24,2341,449.310000,22,0.009398,0.318182
3,12349,18,1,1757.55,631,1757.550000,73,0.115689,0.000000
4,12350,309,1,334.40,197,334.400000,17,0.086294,0.000000
5,12352,35,8,2506.04,536,313.255000,59,0.110075,0.322034
6,12353,203,1,89.00,20,89.000000,4,0.200000,0.000000
7,12354,231,1,1079.40,530,1079.400000,58,0.109434,0.000000
8,12355,213,1,459.40,240,459.400000,13,0.054167,0.000000
9,12356,22,3,2811.43,1591,937.143333,53,0.033312,0.075472
